# Agentic Phishing Email Detection and Explanation

Given a raw email, this notebook returns a **verdict** (phishing / suspicious / legitimate),
a **risk score** (0-100), **reasons** backed by evidence, and a **recommended action**.

Three agents work together:
- **Technical Analyst**: deterministic Python checks on headers and URLs (no LLM on a normal run).
- **Content Analyst**: an LLM that reads the text and identifies intent and social-engineering tactics.
- **Verdict and Explainer**: merges both reports. If they disagree, it asks one analyst a single follow-up question.

Run the cells top to bottom. Each stage ends with a **checkpoint** cell that tells you whether the stage worked.

## Stage 0: Config and setup

Everything you might want to change lives in the next cell. No other cell hardcodes model names or limits.

In [ ]:
# ---- Config: the only place to change models, limits and paths ----
PRIMARY_MODEL  = "gemini-3.1-flash-lite"   # used for ALL evaluation runs
FALLBACK_MODEL = "gemini-3.5-flash-lite"   # used only if the primary fails the smoke test

RPM_LIMIT       = 15     # requests per minute on the free tier
RPD_LIMIT       = 500    # requests per day (resets at midnight Pacific)
RPD_SAFETY_STOP = 450    # stop evaluation runs here to leave headroom

SEED             = 42
PROMPT_VERSION   = "v1"  # bump this when a prompt changes, so old cache entries are ignored
BODY_CHAR_LIMIT  = 6000  # email bodies are truncated to this before going to the LLM

# Score thresholds that turn the risk score into a label
PHISHING_THRESHOLD   = 70   # score >= 70 -> phishing
SUSPICIOUS_THRESHOLD = 40   # 40-69 -> suspicious, below 40 -> legitimate

import os, sys
IN_COLAB = "google.colab" in sys.modules
# Colab keeps data, cache and results on Google Drive. A local run keeps them in a
# git-ignored "runtime" folder next to the notebook.
DRIVE_ROOT = "/content/drive/MyDrive/phish_project" if IN_COLAB else os.path.abspath("runtime")

CREWAI_VERSION = "1.15.28"

# The Stage 1 smoke test costs about 3 API requests. It runs by default; the local
# runner script turns it off (RUN_SMOKE_TEST=0) so repeated dev runs don't spend quota.
RUN_SMOKE_TEST = os.environ.get("RUN_SMOKE_TEST", "1") == "1"

### Install

CrewAI is pinned to one version, with its `google-genai` extra so it can talk to Gemini directly.
It needs Python 3.10–3.13. If pip says it changed packages that were already loaded,
use **Runtime → Restart session**, then run the cells again from the top. (The install is quick the second time.)

In [ ]:
import subprocess
print("Python", sys.version.split()[0], "| running in", "Colab" if IN_COLAB else "a local Jupyter kernel")
assert (3, 10) <= sys.version_info[:2] < (3, 14), "CrewAI 1.15.x needs Python 3.10-3.13"

# Locally, the packages come from requirements.txt in the project venv instead.
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           f"crewai[google-genai]=={CREWAI_VERSION}"])

In [ ]:
# Turn off CrewAI's anonymous telemetry; nothing about our runs needs to leave the machine except the Gemini calls.
os.environ["CREWAI_TELEMETRY_OPT_OUT"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"

import crewai, google.genai, pydantic
print("crewai      ", crewai.__version__)
print("google-genai", google.genai.__version__)
print("pydantic    ", pydantic.__version__)

# Persistent storage for the cache and results, so an interrupted run can resume.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
for sub in ["data", "cache", "results"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)
print("Storage folder:", DRIVE_ROOT)

# The API key never appears in a cell.
# In Colab it comes from Colab Secrets (key icon in the left sidebar).
# Locally it comes from a git-ignored .env file next to the notebook, with one line: GEMINI_API_KEY=...
if IN_COLAB:
    from google.colab import userdata
    _key = userdata.get("GEMINI_API_KEY")
else:
    _key = os.environ.get("GEMINI_API_KEY")
    if not _key and os.path.exists(".env"):
        for line in open(".env", encoding="utf-8"):
            name, _, value = line.strip().partition("=")
            if name.strip() == "GEMINI_API_KEY":
                _key = value.strip().strip('"')
assert _key, "No GEMINI_API_KEY found (Colab Secrets, or .env when running locally)"
os.environ["GEMINI_API_KEY"] = _key
os.environ["GOOGLE_API_KEY"] = _key   # some libraries look for this name instead
del _key                              # don't leave it lying around in a variable
print("key loaded:", bool(os.environ.get("GEMINI_API_KEY")))

**✅ Checkpoint 0:** the cell above prints three version numbers, the storage folder, and `key loaded: True`.
It must not print the key itself.

## Stage 1: Smoke test

This is the riskiest part of the project, so it runs before anything else is built. It tests three things:
1. The model names in the config exist on your account.
2. A plain call through CrewAI's Gemini integration works.
3. A CrewAI agent can return **structured output** (a pydantic object), on both the primary and the fallback model.

This stage uses about 3 requests of your daily quota.

In [ ]:
# 1. Do the configured model names exist? Listing models does not use generation quota.
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
available = sorted(m.name.removeprefix("models/") for m in client.models.list())

print("Flash Lite models on this key:")
for name in available:
    if "flash-lite" in name:
        print("  ", name)

for name in [PRIMARY_MODEL, FALLBACK_MODEL]:
    print(f"{name}: {'found' if name in available else 'NOT FOUND, fix the config cell'}")

In [ ]:
# 2. One plain call through CrewAI's native Gemini LLM.
from crewai import LLM

def make_llm(model_name):
    # temperature=0 makes answers as repeatable as the model allows
    return LLM(model=f"gemini/{model_name}", temperature=0)

if RUN_SMOKE_TEST:
    reply = make_llm(PRIMARY_MODEL).call("Reply with the single word OK.")
    print("Reply:", repr(reply))
else:
    print("Smoke test switched off (RUN_SMOKE_TEST is False).")

### The Content Analyst's output schema

The Content Analyst must return this structure instead of free text. Writing it as a pydantic model
means Python checks every field and type for us. The same schema is used in the real system later.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

Intent = Literal["credential_theft", "payment_request", "personal_info", "malware_delivery",
                 "advance_fee", "account_verification", "benign", "other"]
TacticName = Literal["urgency", "authority", "fear", "reward", "scarcity", "curiosity"]

class Tactic(BaseModel):
    tactic: TacticName
    quote: str = Field(description="exact words copied from the email that show this tactic")
    explanation: str

class ContentReport(BaseModel):
    intent: Intent
    tactics: list[Tactic]
    content_risk: int = Field(ge=0, le=100, description="0 = clearly harmless, 100 = clearly malicious")
    summary: str

In [ ]:
# 3. A single agent and task that must return a ContentReport.
from crewai import Agent, Task, Crew
from concurrent.futures import ThreadPoolExecutor

def run_crew(crew, inputs):
    """Colab already runs an asyncio event loop, and CrewAI refuses to start a normal
    (synchronous) kickoff inside one. A separate worker thread has no event loop,
    so we run the kickoff there and wait for its result."""
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(crew.kickoff, inputs=inputs).result()

# Synthetic test email, written for this notebook. It is not a real person's email.
SMOKE_EMAIL = """Subject: Urgent: your account will be suspended

Dear customer,

We detected unusual sign-in activity on your account. To avoid permanent suspension,
verify your identity within 24 hours by logging in here: http://secure-paypa1-login.xyz/verify

Thank you,
Security Team"""

def smoke_test(model_name):
    analyst = Agent(
        role="Content Analyst",
        goal="Identify what an email is trying to make the reader do, and which manipulation tactics it uses.",
        backstory="You study phishing emails and describe them precisely, quoting the email as evidence.",
        llm=make_llm(model_name),
        allow_delegation=False,   # no handing work to other agents
        max_iter=1,               # one attempt, so one task = one request
        verbose=False,
    )
    task = Task(
        description=("Analyse this email. Quote its exact words as evidence for each tactic.\n\n"
                     "EMAIL:\n{email}"),
        expected_output="A ContentReport.",
        agent=analyst,
        output_pydantic=ContentReport,
    )
    result = run_crew(Crew(agents=[analyst], tasks=[task], verbose=False), {"email": SMOKE_EMAIL})

    print(f"=== {model_name} ===")
    print("Token usage:", result.token_usage)
    if result.pydantic is None:
        print("Structured output FAILED. Raw reply:")
        print(result.raw)
        return None
    print(result.pydantic.model_dump_json(indent=2))
    return result.pydantic

smoke_results = {}
for name in [PRIMARY_MODEL, FALLBACK_MODEL] if RUN_SMOKE_TEST else []:
    try:
        smoke_results[name] = smoke_test(name)
    except Exception as e:   # show the error but keep going, so both models get tested
        print(f"=== {name} === ERROR: {type(e).__name__}: {e}")
        smoke_results[name] = None

In [ ]:
# ✅ Checkpoint 1
for name, report in smoke_results.items():
    print(f"{name}: {'PASS' if isinstance(report, ContentReport) else 'FAIL'}")

if not RUN_SMOKE_TEST:
    print("Smoke test switched off; nothing to check.")
elif isinstance(smoke_results.get(PRIMARY_MODEL), ContentReport):
    print("\nPrimary model works. Keep it for every evaluation run.")
else:
    print("\nPrimary model failed. Stop here and decide on a fallback before building further.")

**✅ Checkpoint 1:** the model names are found, the plain call replies with OK,
and the primary model shows `PASS` with a sensible report (intent `credential_theft`, tactics such as urgency and fear,
quotes copied from the email). Note the token usage. It feeds the budget estimate.

## Stage 2: Data acquisition and licensing

We need public emails **with full headers**, because the Technical Analyst reads headers.
The sources were checked on 2026-10-22:

| Source | Label | Years | Headers | License / terms |
|---|---|---|---|---|
| [Nazario phishing corpus](https://monkey.org/~jose/phishing/) (2023, 2024, 2025 mailboxes) | phishing | 2023–2025 | full | CC-BY-4.0 (attribution: Jose Nazario) |
| [SpamAssassin public corpus](https://spamassassin.apache.org/old/publiccorpus/) (20030228 easy_ham, easy_ham_2, hard_ham) | legitimate | ~2002 | full | No formal license. Offered publicly for testing spam filters. Copyright stays with the original senders. Not redistributed here. |

Sources considered and **not** used:
- **SpamAssassin spam** is spam, not phishing.
- **Enron raw** is 443 MB of 2001 internal mail with no auth headers.
- **The Zenodo curated CSVs** (CC-BY-4.0) have no full headers, label spam rather than phishing, and their Enron copy is lowercased and tokenised.

**Known limitation (era gap).** The phishing is from 2023–2025, while the legitimate mail is from about 2002.
A model can tell them apart by date formats, header styles, or the presence of `Authentication-Results`
headers (modern mail has them, 2002 mail doesn't), rather than by phishing signals. To reduce this:
- the baseline and the LLM-only config get the email with date, routing and auth headers removed;
- the Technical Analyst scores auth headers only when they show an outright `fail`, never just because they are present or missing.

This limitation is stated in the report.

Raw files are downloaded once into Drive. Later runs reuse them. The Nazario server is slow (expect a few minutes).

In [ ]:
import urllib.request, tarfile, mailbox, hashlib
import pandas as pd

RAW_DIR = f"{DRIVE_ROOT}/data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

NAZARIO_FILES = ["phishing-2023", "phishing-2024", "phishing-2025"]
SPAMASSASSIN_FILES = ["20030228_easy_ham.tar.bz2", "20030228_easy_ham_2.tar.bz2", "20030228_hard_ham.tar.bz2"]

def download(url, dest):
    """Download url to dest, unless it is already there."""
    if os.path.exists(dest):
        print("already have", os.path.basename(dest))
        return
    print("downloading", url)
    urllib.request.urlretrieve(url, dest + ".part")
    os.replace(dest + ".part", dest)   # only a complete download gets the real name

for name in NAZARIO_FILES:
    download(f"https://monkey.org/~jose/phishing/{name}", f"{RAW_DIR}/{name}")
for name in SPAMASSASSIN_FILES:
    download(f"https://spamassassin.apache.org/old/publiccorpus/{name}", f"{RAW_DIR}/{name}")

In [ ]:
def make_email_id(raw):
    """Stable ID: the same email text always gets the same ID, across runs and machines."""
    normalised = "\n".join(line.rstrip() for line in raw.replace("\r\n", "\n").split("\n")).strip()
    return hashlib.sha256(normalised.encode("utf-8", errors="replace")).hexdigest()[:16]

def to_text(raw_bytes):
    # Emails are mostly ASCII. "replace" keeps odd bytes from crashing the parse.
    return raw_bytes.decode("utf-8", errors="replace")

rows = []

# Nazario: each file is an mbox (many emails in one file).
for name in NAZARIO_FILES:
    for msg in mailbox.mbox(f"{RAW_DIR}/{name}"):
        if "FOLDER INTERNAL DATA" in str(msg.get("Subject", "")):
            continue   # mail-client bookkeeping record at the top of each file, not an email
        rows.append({"source": f"nazario_{name[-4:]}", "label": 1, "raw": to_text(msg.as_bytes())})

# SpamAssassin: each archive holds one email per file.
for name in SPAMASSASSIN_FILES:
    with tarfile.open(f"{RAW_DIR}/{name}", "r:bz2") as tar:
        for member in tar.getmembers():
            if not member.isfile() or member.name.endswith("cmds"):
                continue   # "cmds" is an index file, not an email
            raw = to_text(tar.extractfile(member).read())
            rows.append({"source": "spamassassin_" + name.split("_", 1)[1].removesuffix(".tar.bz2"),
                         "label": 0, "raw": raw})

emails = pd.DataFrame(rows)
emails["email_id"] = emails["raw"].map(make_email_id)
emails["has_headers"] = emails["raw"].str.contains(r"(?m)^From:", regex=True)   # (?m): ^ matches at every line start
emails["synthetic"] = False
emails = emails[["email_id", "source", "label", "has_headers", "synthetic", "raw"]]

emails.to_parquet(f"{DRIVE_ROOT}/data/emails.parquet", index=False)
print("saved", len(emails), "emails")

In [ ]:
# ✅ Checkpoint 2
print(emails.groupby(["source", "label"]).size().rename("count").to_string(), "\n")
print("phishing:", (emails.label == 1).sum(), "  legitimate:", (emails.label == 0).sum())
print("without headers:", (~emails.has_headers).sum())
print("duplicate IDs (exact copies, removed in Stage 3):", emails.email_id.duplicated().sum(), "\n")

# One parsed sample of each class, to check the parsing looks right
import email
from email import policy
for label in [1, 0]:
    row = emails[emails.label == label].iloc[0]
    msg = email.message_from_string(row.raw, policy=policy.default)
    print(f"--- label {label} ({row.source}, id {row.email_id})")
    print("From:   ", msg["From"])
    print("Subject:", msg["Subject"])
    body = msg.get_body(preferencelist=("plain", "html"))
    print(body.get_content()[:300] if body else "(no text body)", "\n")

**✅ Checkpoint 2:** a count table per source; roughly 1,000+ phishing and about 4,100 legitimate emails;
`without headers` close to 0; and two samples whose From/Subject look like real emails.
The classes are unbalanced, which is fine: Stage 3 samples a balanced 50/50 test subset.

## Stage 3: Dedupe, split, sample

Phishing campaigns send the same email to many people, so a corpus often holds copies that differ only in
spacing, case, or a tracking number. If one copy lands in training and another in testing, the test score is inflated.
So we remove duplicates **before** splitting.

Two emails count as duplicates when their **body text** is the same after normalising: lower-case, digits removed
(which drops tracking numbers and dates), and punctuation and spacing collapsed. This catches copies, not rewrites.

Then:
- **Train / held-out split, 80 / 20.** Stratified by source, so every source appears in both parts in the same proportion. The baseline trains on the train part only.
- **Test subset, 100 emails from the held-out part.** 50 phishing and 50 legitimate, stratified by source. These are the emails all four systems are compared on.
- **The IDs of every split are saved**, so every later run uses exactly the same emails.

In [ ]:
import re, json
import email
from email import policy
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split

def get_body_text(raw):
    """The readable text of an email: the plain-text part if there is one, otherwise the HTML part converted to text."""
    try:
        msg = email.message_from_string(raw, policy=policy.default)
        part = msg.get_body(preferencelist=("plain", "html"))
        if part is None:
            return ""
        content = part.get_content()
        if part.get_content_type() == "text/html":
            content = BeautifulSoup(content, "html.parser").get_text(" ")
        return content
    except Exception:   # badly formed emails exist in every corpus; treat them as having no readable body
        return ""

def body_fingerprint(raw):
    text = get_body_text(raw).lower()
    text = re.sub(r"\d+", "", text)          # drop digits: tracking numbers, dates, amounts
    text = re.sub(r"[\W_]+", " ", text)      # punctuation and spacing -> single space
    return hashlib.sha256(text.strip().encode()).hexdigest()

emails = pd.read_parquet(f"{DRIVE_ROOT}/data/emails.parquet")
emails["body_text"] = emails["raw"].map(get_body_text)
emails["fingerprint"] = emails["raw"].map(body_fingerprint)

empty = emails["body_text"].str.strip() == ""
print("emails with no readable body (dropped):", empty.sum())
emails = emails[~empty]

before = len(emails)
emails = emails.drop_duplicates(subset="fingerprint", keep="first")
print("duplicates removed:", before - len(emails))
print("emails left:", len(emails))

In [ ]:
train, heldout = train_test_split(emails, test_size=0.2, stratify=emails["source"], random_state=SEED)

def take_balanced(pool, n):
    """n emails from pool, keeping each source's share the same as in pool."""
    picked, _ = train_test_split(pool, train_size=n, stratify=pool["source"], random_state=SEED)
    return picked

subset = pd.concat([take_balanced(heldout[heldout.label == 1], 50),
                    take_balanced(heldout[heldout.label == 0], 50)])

# Demo emails, picked by reading the subset (choosing examples is not labelling: the labels come from the corpora).
# They come from the subset, so their evaluation results double as the demo's cached backup.
DEMO_IDS = {
    # "eAlert M&T Bank" sent from stoporgo.org, link to angla.cc. SPF and DKIM pass,
    # which shows that passing authentication only proves who sent it, not that it's safe.
    "obvious_phish": "7deba0cab6553440",
    # Sent through the real Google Drive (DKIM, SPF and DMARC all pass, link is drive.google.com),
    # but the display name uses look-alike Cyrillic letters ("Wеllsfаrgo") and Reply-To goes to a .eu domain.
    "subtle_phish": "a62c055b1cf2435e",
    # A genuine 2002 security-advisory mailing list: security wording and links, but consistent domains.
    "legitimate": "12865f8f440c21cc",
}

splits = {"train": train.email_id.tolist(),
          "heldout": heldout.email_id.tolist(),
          "subset": subset.email_id.tolist(),
          "demo": DEMO_IDS}
with open(f"{DRIVE_ROOT}/data/splits.json", "w") as f:
    json.dump(splits, f)

# Keep the deduplicated table too; later stages load it with the splits.
emails.drop(columns=["fingerprint"]).to_parquet(f"{DRIVE_ROOT}/data/emails_dedup.parquet", index=False)

In [ ]:
# ✅ Checkpoint 3
balance = pd.DataFrame({
    "train":   train.groupby("source").size(),
    "heldout": heldout.groupby("source").size(),
    "subset":  subset.groupby("source").size(),
}).fillna(0).astype(int)
print(balance, "\n")
print("subset labels:", subset.label.value_counts().to_dict())

overlap = set(splits["train"]) & set(splits["heldout"])
print("train/held-out overlap:", len(overlap))
print("subset inside held-out:", set(splits["subset"]) <= set(splits["heldout"]))
assert not overlap and set(splits["subset"]) <= set(splits["heldout"])

demo_ok = set(DEMO_IDS.values()) <= set(splits["subset"])
print("demo emails inside subset:", demo_ok)
assert demo_ok, "a demo ID is no longer in the subset (did the data or SEED change?)"
print("saved:", f"{DRIVE_ROOT}/data/splits.json")

**✅ Checkpoint 3:** a balance table where every source appears in train, held-out and subset;
the subset is exactly 50 phishing and 50 legitimate; the overlap is 0; the 3 demo emails are inside the subset.

Things noticed while reading the subset, for the report:
- A few phishing emails are in Spanish or German. Multilingual phishing is out of scope, but these are kept because they are part of the labelled corpus.
- Many phishing emails target the corpus owner's own domain (monkey.org), often with a forged `From:` on that domain.
- Most legitimate emails are 2002 technical mailing-list posts, a narrower kind of mail than a normal inbox.